<a href="https://colab.research.google.com/github/vzuza/codigos_amanha/blob/main/projeto_codigos_pessoal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [100]:
from google.colab import drive
from IPython.display import display
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
import matplotlib as mpl
import geopandas as gpd
import zipfile
import os
import re
import unicodedata
import requests
import geopandas as gpd
import folium


In [19]:
#Caminho dos dados no drive (IBGE)

drive.mount('/content/drive')

PASTA_IBGE = '/content/drive/MyDrive/Codigos_Projeto/IBGE'

caminho_alfabetizacao = os.path.join(
    PASTA_IBGE, 'alfabetizacao_BR.xlsx'
)

caminho_domicilio1 = os.path.join(
    PASTA_IBGE, 'caracteristicas_domicilio1_BR.xlsx'
)

caminho_domicilio2 = os.path.join(
    PASTA_IBGE, 'caracteristicas_domicilio2_BR.xlsx'
)

caminho_demografia = os.path.join(
    PASTA_IBGE, 'demografia_BR.xlsx'
)

caminho_renda = os.path.join(
    PASTA_IBGE, 'renda_PR.csv'
)

arquivos = {
    'Alfabetização': caminho_alfabetizacao,
    'Domicílio 1': caminho_domicilio1,
    'Domicílio 2': caminho_domicilio2,
    'Demografia': caminho_demografia,
    'Renda': caminho_renda
}

df_alfabetizacao_BR = pd.read_excel(caminho_alfabetizacao)
df_domicilio1_BR = pd.read_excel(caminho_domicilio1)
df_domicilio2_BR = pd.read_excel(caminho_domicilio2)
df_demografia_BR = pd.read_excel(caminho_demografia)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
#API renda IBGE

url = (
    'https://apisidra.ibge.gov.br/values/'
    't/10295/n6/all/v/13534/p/2022/'
    'c2/6794/c86/95251/c58/95253'
)


resposta = requests.get(url, timeout=120)
resposta.raise_for_status()

dados = resposta.json()


df_renda_PR = pd.DataFrame(dados[1:]).rename(
    columns=dados[0]
)

print(df_renda_PR.columns.tolist())

col_codigo = 'Município (Código)'


df_renda_PR = df_renda_PR[
    df_renda_PR[col_codigo].astype(str).str.startswith('41')
].copy()


df_renda_PR.to_csv(
    'renda_municipios_PR.csv',
    index=False,
    encoding='utf-8-sig'
)


['Nível Territorial (Código)', 'Nível Territorial', 'Unidade de Medida (Código)', 'Unidade de Medida', 'Valor', 'Município (Código)', 'Município', 'Variável (Código)', 'Variável', 'Ano (Código)', 'Ano', 'Sexo (Código)', 'Sexo', 'Cor ou raça (Código)', 'Cor ou raça', 'Grupo de idade (Código)', 'Grupo de idade']


In [21]:
# Caminho dos dados no drive (S2ID)

PASTA_S2ID = '/content/drive/MyDrive/Codigos_Projeto/S2ID'

caminho_s2id = os.path.join(
    PASTA_S2ID, 'S2ID_PR_2015_2025.csv'
)

df_s2id = pd.read_csv(
    caminho_s2id,
    encoding='utf-8-sig'
)

print(f'A base de dados possui {df_s2id.shape[0]} linhas e {df_s2id.shape[1]} colunas')

A base de dados possui 1157 linhas e 55 colunas


In [22]:
#Limpando os dados do IBGE BR para IBGE PR

df_alfabetizacao_PR = df_alfabetizacao_BR[
    df_alfabetizacao_BR['CD_MUN'].astype(str).str.startswith('41')
].copy()
print(f'A base de dados possui {df_alfabetizacao_PR.shape[0]} linhas e {df_alfabetizacao_PR.shape[1]} colunas')

df_domicilio1_PR = df_domicilio1_BR[
    df_domicilio1_BR['CD_MUN'].astype(str).str.startswith('41')
].copy()

print(f'A base de dados possui {df_domicilio1_PR .shape[0]} linhas e {df_domicilio1_PR .shape[1]} colunas')

df_domicilio2_PR = df_domicilio2_BR[
    df_domicilio2_BR['CD_MUN'].astype(str).str.startswith('41')
].copy()
print(f'A base de dados possui {df_domicilio2_PR.shape[0]} linhas e {df_domicilio2_PR.shape[1]} colunas')

df_demografia_PR = df_demografia_BR[
    df_demografia_BR['CD_MUN'].astype(str).str.startswith('41')
].copy()
print(f'A base de dados possui {df_demografia_PR.shape[0]} linhas e {df_demografia_PR.shape[1]} colunas')

A base de dados possui 399 linhas e 364 colunas
A base de dados possui 399 linhas e 91 colunas
A base de dados possui 399 linhas e 408 colunas
A base de dados possui 399 linhas e 38 colunas


In [23]:
#Padronizando os dados de alfabetização (IBGE/PR)

df_alfabetizacao_limpo = df_alfabetizacao_PR[
    ['CD_MUN', 'NM_MUN', 'V00900', 'V00901']
].copy()

df_alfabetizacao_limpo = df_alfabetizacao_limpo.rename(columns={
    'CD_MUN': 'codigo_municipio',
    'NM_MUN': 'municipio',
    'V00900': 'populacao_alfabetizada_15mais',
    'V00901': 'populacao_nao_alfabetizada_15mais'
})

df_alfabetizacao_limpo.head()
print(f'A planilha de alfabetização possui {df_alfabetizacao_limpo.shape[0]} linhas e {df_alfabetizacao_limpo.shape[1]} colunas')


A planilha de alfabetização possui 399 linhas e 4 colunas


In [24]:
#Padronizando os dados de demografia (IBGE/PR)

df_demografia_limpo = df_demografia_PR[
    [
        'CD_MUN',
        'NM_MUN',
        'V01007',
        'V01008',
        'V01031',
        'V01032',
        'V01033',
        'V01040',
        'V01041'
    ]
].copy()

df_demografia_limpo = df_demografia_limpo.rename(columns={
    'CD_MUN': 'codigo_municipio',
    'NM_MUN': 'municipio',
    'V01007': 'populacao_masculina',
    'V01008': 'populacao_feminina',
    'V01031': 'populacao_0_4_anos',
    'V01032': 'populacao_5_9_anos',
    'V01033': 'populacao_10_14_anos',
    'V01040': 'populacao_60_69_anos',
    'V01041': 'populacao_70_mais_anos'
})

df_demografia_limpo.head()
print(f'A planilha de demografia possui {df_demografia_limpo.shape[0]} linhas e {df_demografia_limpo.shape[1]} colunas')


A planilha de demografia possui 399 linhas e 9 colunas


In [25]:
#Padronizando os dados das características de domicilio 1 e 2 / saneamento (IBGE/PR)

# características de domicilio 2

df_saneamento_limpo = df_domicilio2_PR[
    [
        'CD_MUN',
        'NM_MUN',
        'V00312',
        'V00313',
        'V00314',
        'V00315',
        'V00316'
    ]
].copy()

df_saneamento_limpo = df_saneamento_limpo.rename(columns={
    'CD_MUN': 'codigo_municipio',
    'NM_MUN': 'municipio',
    'V00312': 'domicilios_fossa_rudimentar',
    'V00313': 'domicilios_vala',
    'V00314': 'domicilios_rio_lago_mar',
    'V00315': 'domicilios_outra_forma',
    'V00316': 'domicilios_sem_banheiro_sanitario'
})

# características de domicilio 1

df_total_domicilios = df_domicilio1_PR[
    ['CD_MUN', 'V00001']
].copy()

df_total_domicilios = df_total_domicilios.rename(columns={
    'CD_MUN': 'codigo_municipio',
    'V00001': 'total_domicilios'
})


df_saneamento_limpo['codigo_municipio'] = (
    df_saneamento_limpo['codigo_municipio'].astype(str)
)

df_total_domicilios['codigo_municipio'] = (
    df_total_domicilios['codigo_municipio'].astype(str)
)


#juntando as tabelas

df_saneamento_limpo = df_saneamento_limpo.merge(
    df_total_domicilios,
    on='codigo_municipio',
    how='left',
    validate='one_to_one'
)

df_saneamento_limpo.head()
print(f'A planilha de saneamento possui {df_saneamento_limpo.shape[0]} linhas e {df_saneamento_limpo.shape[1]} colunas')

A planilha de saneamento possui 399 linhas e 8 colunas


In [26]:
#Padronizando os dados de renda (IBGE/PR)

df_renda_PR = df_renda_PR[
    ['Município (Código)', 'Município', 'Valor']
].copy()

df_renda_PR = df_renda_PR.rename(columns={
    'Município (Código)': 'COD',
    'Município': 'MUN',
    'Valor': 'RENDA_DOMICILIAR_PER_CAPITA'
})

df_renda_PR['COD'] = df_renda_PR['COD'].astype(str)

df_renda_PR['RENDA_DOMICILIAR_PER_CAPITA'] = pd.to_numeric(
    df_renda_PR['RENDA_DOMICILIAR_PER_CAPITA'],
    errors='coerce'
)

display(df_renda_PR.head())

,COD,MUN,RENDA_DOMICILIAR_PER_CAPITA
3912,4100103,Abatiá - PR,920.00
3913,4100202,Adrianópolis - PR,808.00
3914,4100301,Agudos do Sul - PR,1000.00
3915,4100400,Almirante Tamandaré - PR,1066.67
3916,4100459,Altamira do Paraná - PR,978.00


In [27]:
#Modelando os dados de analfabetismo IBGE

df_alfabetizacao_limpo['populacao_alfabetizada_15mais'] = pd.to_numeric(
    df_alfabetizacao_limpo['populacao_alfabetizada_15mais'],
    errors='coerce'
)

df_alfabetizacao_limpo['populacao_nao_alfabetizada_15mais'] = pd.to_numeric(
    df_alfabetizacao_limpo['populacao_nao_alfabetizada_15mais'],
    errors='coerce'
)

df_alfabetizacao_limpo['taxa_analfabetismo'] = (
    df_alfabetizacao_limpo['populacao_nao_alfabetizada_15mais'] /
    (
        df_alfabetizacao_limpo['populacao_alfabetizada_15mais'] +
        df_alfabetizacao_limpo['populacao_nao_alfabetizada_15mais']
    )
) * 100

df_alfabetizacao_limpo.head()
df_alfabetizacao_limpo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 399 entries, 3912 to 4310
Data columns (total 5 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   codigo_municipio                   399 non-null    int64  
 1   municipio                          399 non-null    object 
 2   populacao_alfabetizada_15mais      399 non-null    int64  
 3   populacao_nao_alfabetizada_15mais  399 non-null    int64  
 4   taxa_analfabetismo                 399 non-null    float64
dtypes: float64(1), int64(3), object(1)
memory usage: 18.7+ KB


In [28]:
#Modelando os dados de demografia (IBGE/PR)

colunas_numericas = [
    'populacao_masculina',
    'populacao_feminina',
    'populacao_0_4_anos',
    'populacao_5_9_anos',
    'populacao_10_14_anos',
    'populacao_60_69_anos',
    'populacao_70_mais_anos'
]

for coluna in colunas_numericas:
    df_demografia_limpo[coluna] = pd.to_numeric(
        df_demografia_limpo[coluna],
        errors='coerce'
    )

#População total

df_demografia_limpo['populacao_total'] = (
    df_demografia_limpo['populacao_masculina'] +
    df_demografia_limpo['populacao_feminina']
)

# % população de 0 a 14 anos

df_demografia_limpo['populacao_0_14_anos'] = (
    df_demografia_limpo['populacao_0_4_anos'] +
    df_demografia_limpo['populacao_5_9_anos'] +
    df_demografia_limpo['populacao_10_14_anos']
)

df_demografia_limpo['percentual_0_14_anos'] = (
    df_demografia_limpo['populacao_0_14_anos'] /
    df_demografia_limpo['populacao_total']
) * 100

# % população de 65 anos ou mais

df_demografia_limpo['populacao_60_mais_anos'] = (
    df_demografia_limpo['populacao_60_69_anos'] +
    df_demografia_limpo['populacao_70_mais_anos']
)

df_demografia_limpo['percentual_60_mais_anos'] = (
    df_demografia_limpo['populacao_60_mais_anos'] /
    df_demografia_limpo['populacao_total']
) * 100

df_demografia_limpo.head()

,codigo_municipio,municipio,populacao_masculina,populacao_feminina,populacao_0_4_anos,populacao_5_9_anos,populacao_10_14_anos,populacao_60_69_anos,populacao_70_mais_anos,populacao_total,populacao_0_14_anos,percentual_0_14_anos,populacao_60_mais_anos,percentual_60_mais_anos
3912,4100103,Abatiá,3599,3634,468,473,439,858,697,7233,1380,19.079220,1555,21.498687
3913,4100202,Adrianópolis,3139,3073,386,416,477,634,555,6212,1279,20.589182,1189,19.140373
3914,4100301,Agudos do Sul,5275,4958,704,755,725,1064,709,10233,2184,21.342715,1773,17.326297
3915,4100400,Almirante Tamandaré,59028,60797,8185,10003,8909,8511,4988,119825,27097,22.613812,13499,11.265596
3916,4100459,Altamira do Paraná,1818,1772,209,246,221,383,311,3590,676,18.830084,694,19.331476


In [29]:
#Modelando os dados de saneamento (IBGE/PR)

colunas_numericas = [
    'total_domicilios',
    'domicilios_fossa_rudimentar',
    'domicilios_vala',
    'domicilios_rio_lago_mar',
    'domicilios_outra_forma',
    'domicilios_sem_banheiro_sanitario'
]

for coluna in colunas_numericas:
    df_saneamento_limpo[coluna] = pd.to_numeric(
        df_saneamento_limpo[coluna],
        errors='coerce'
    )

df_saneamento_limpo['domicilios_saneamento_inadequado'] = (
    df_saneamento_limpo[
        [
            'domicilios_fossa_rudimentar',
            'domicilios_vala',
            'domicilios_rio_lago_mar',
            'domicilios_outra_forma',
            'domicilios_sem_banheiro_sanitario'
        ]
    ].sum(axis=1, min_count=5)
)

df_saneamento_limpo['percentual_saneamento_inadequado'] = (
    df_saneamento_limpo['domicilios_saneamento_inadequado'] /
    df_saneamento_limpo['total_domicilios'].replace(0, np.nan)
) * 100


df_saneamento_limpo.head()

,codigo_municipio,municipio,domicilios_fossa_rudimentar,domicilios_vala,domicilios_rio_lago_mar,domicilios_outra_forma,domicilios_sem_banheiro_sanitario,total_domicilios,domicilios_saneamento_inadequado,percentual_saneamento_inadequado
0,4100103,Abatiá,698,4,34,0,0,2711,736,27.148654
1,4100202,Adrianópolis,749,48,90,6,0,2304,893,38.758681
2,4100301,Agudos do Sul,2369,3,0,4,0,3633,2376,65.400495
3,4100400,Almirante Tamandaré,4873,167,365,40,0,40796,5445,13.346897
4,4100459,Altamira do Paraná,1089,0,0,0,0,1330,1089,81.879699


In [30]:
# Modelando os dados de renda (IBGE/PR)

renda_min = df_renda_PR['RENDA_DOMICILIAR_PER_CAPITA'].min()
renda_max = df_renda_PR['RENDA_DOMICILIAR_PER_CAPITA'].max()

df_renda_PR['VULNERABILIDADE_RENDA'] = (
    (renda_max - df_renda_PR['RENDA_DOMICILIAR_PER_CAPITA']) /
    (renda_max - renda_min)
)

In [31]:
# Padronização e integração dos códigos do IBGE (df_vulnerabilidade)

bases = [
    df_demografia_limpo,
    df_alfabetizacao_limpo,
    df_saneamento_limpo
]

for df in bases:
    df['codigo_municipio'] = (
        df['codigo_municipio']
        .astype(str)
        .str.replace(r'\.0$', '', regex=True)
        .str.zfill(7)
    )

df_renda_PR['COD'] = (
    df_renda_PR['COD']
    .astype(str)
    .str.zfill(7)
)


df_vulnerabilidade = df_demografia_limpo[
    ['codigo_municipio', 'municipio',
     'percentual_0_14_anos',
     'percentual_60_mais_anos']
].copy()


df_vulnerabilidade = df_vulnerabilidade.merge(
    df_alfabetizacao_limpo[
        ['codigo_municipio', 'taxa_analfabetismo']
    ],
    on='codigo_municipio',
    how='left',
    validate='one_to_one'
)

df_vulnerabilidade = df_vulnerabilidade.merge(
    df_saneamento_limpo[
        ['codigo_municipio',
         'percentual_saneamento_inadequado']
    ],
    on='codigo_municipio',
    how='left',
    validate='one_to_one'
)


df_vulnerabilidade = df_vulnerabilidade.merge(
    df_renda_PR[
        ['COD', 'RENDA_DOMICILIAR_PER_CAPITA']
    ],
    left_on='codigo_municipio',
    right_on='COD',
    how='left',
    validate='one_to_one'
)

df_vulnerabilidade.drop(columns='COD', inplace=True)

print('Dimensões:', df_vulnerabilidade.shape)
print('Municípios:', df_vulnerabilidade['codigo_municipio'].nunique())

display(df_vulnerabilidade.head())

Dimensões: (399, 7)
Municípios: 399


,codigo_municipio,municipio,percentual_0_14_anos,percentual_60_mais_anos,taxa_analfabetismo,percentual_saneamento_inadequado,RENDA_DOMICILIAR_PER_CAPITA
0,4100103,Abatiá,19.079220,21.498687,11.010429,27.148654,920.00
1,4100202,Adrianópolis,20.589182,19.140373,12.461758,38.758681,808.00
2,4100301,Agudos do Sul,21.342715,17.326297,6.689878,65.400495,1000.00
3,4100400,Almirante Tamandaré,22.613812,11.265596,3.490831,13.346897,1066.67
4,4100459,Altamira do Paraná,18.830084,19.331476,10.603981,81.879699,978.00


In [32]:
# Indicadores em que valores maiores significam

# maior vulnerabilidade

indicadores_diretos = [
    'percentual_0_14_anos',
    'percentual_60_mais_anos',
    'taxa_analfabetismo',
    'percentual_saneamento_inadequado'
]

for coluna in indicadores_diretos:

    minimo = df_vulnerabilidade[coluna].min()
    maximo = df_vulnerabilidade[coluna].max()

    df_vulnerabilidade[coluna + '_norm'] = (
        (df_vulnerabilidade[coluna] - minimo) /
        (maximo - minimo)
    )

# Renda: normalização inversa

minimo = df_vulnerabilidade[
    'RENDA_DOMICILIAR_PER_CAPITA'
].min()

maximo = df_vulnerabilidade[
    'RENDA_DOMICILIAR_PER_CAPITA'
].max()

df_vulnerabilidade['renda_norm'] = (
    (maximo - df_vulnerabilidade['RENDA_DOMICILIAR_PER_CAPITA']) /
    (maximo - minimo)
)

In [33]:
#Calcular o componente V

colunas_v = [
    'percentual_0_14_anos_norm',
    'percentual_60_mais_anos_norm',
    'renda_norm',
    'taxa_analfabetismo_norm',
    'percentual_saneamento_inadequado_norm'
]

df_vulnerabilidade['V'] = (
    df_vulnerabilidade[colunas_v].mean(axis=1)
)

display(
    df_vulnerabilidade[
        ['municipio', 'V']
    ].sort_values('V', ascending=False).head(20)
)

,municipio,V
339,São Jerônimo da Serra,0.735358
183,Laranjal,0.730283
307,Rio Branco do Ivaí,0.727577
237,Nova Santa Bárbara,0.723905
126,Grandes Rios,0.723213
105,Espigão Alto do Iguaçu,0.721642
216,Mato Rico,0.715936
395,Doutor Ulysses,0.697681
96,Diamante do Sul,0.696842
160,Itaúna do Sul,0.687533


In [72]:
#Padronizando os dados do S2ID / Excluir doenças infecciosas

df_s2id_naturais = df_s2id_pr.copy()

df_s2id_naturais['COBRADE_codigo'] = (
    df_s2id_naturais['COBRADE']
    .astype(str)
    .str.extract(r'^(\d{5})')[0]
)

codigos_excluir = ['15110', '15120', '15130', '15140']

df_s2id_naturais = df_s2id_naturais[
    df_s2id_naturais['COBRADE_codigo'].str.startswith('1', na=False)
    &
    ~df_s2id_naturais['COBRADE_codigo'].isin(codigos_excluir)
].copy()


Registros naturais: 902
Municípios: 304

COBRADE considerados:
COBRADE
12200 - Enxurradas                                                                                                                                     221
14110 - Estiagem                                                                                                                                       216
13213 - Tempestade Local/Convectiva - Granizo                                                                                                          145
13215 - Tempestade Local/Convectiva - Vendaval                                                                                                         131
13214 - Tempestade Local/Convectiva - Chuvas Intensas                                                                                                   99
12300 - Alagamentos                                                                                                                                     44

In [75]:
# Frequência de desastres naturais por município / Normalização (F)

df_frequencia = (
    df_s2id_naturais
    .groupby('codigo_municipio')
    .agg(
        total_ocorrencias=('Protocolo', 'nunique')
    )
    .reset_index()
)

df_frequencia_completa = (
    df_demografia_limpo[['codigo_municipio', 'municipio']]
    .drop_duplicates('codigo_municipio')
    .merge(df_frequencia, on='codigo_municipio', how='left')
)

df_frequencia_completa['total_ocorrencias'] = (
    df_frequencia_completa['total_ocorrencias'].fillna(0)
)

df_frequencia_completa['frequencia_log'] = np.log1p(
    df_frequencia_completa['total_ocorrencias']
)

min_f = df_frequencia_completa['frequencia_log'].min()
max_f = df_frequencia_completa['frequencia_log'].max()

df_frequencia_completa['F'] = (
    (df_frequencia_completa['frequencia_log'] - min_f)
    / (max_f - min_f)
)


In [79]:
# Impacto de desastres naturais por município / Normalização (I)

#DH
colunas_DH = [
    'DH_Mortos',
    'DH_Feridos',
    'DH_Enfermos',
    'DH_Desabrigados',
    'DH_Desalojados',
    'DH_Desaparecidos',
    'DH_Outros Afetados'
]

#DM
colunas_DM = [
    col for col in df_s2id_naturais.columns
    if col.startswith('DM_') and col.endswith('Valor')
]

# PE
colunas_PE = [
    col for col in df_s2id_naturais.columns
    if col.startswith(('PEPL_', 'PEPR_'))
]

for col in colunas_DH + colunas_DM + colunas_PE:
    df_s2id_naturais[col] = pd.to_numeric(
        df_s2id_naturais[col], errors='coerce'
    ).fillna(0)

df_s2id_naturais['DH_total'] = (
    df_s2id_naturais[colunas_DH].sum(axis=1)
)

df_s2id_naturais['DM_total'] = (
    df_s2id_naturais[colunas_DM].sum(axis=1)
)

df_s2id_naturais['PE_total'] = (
    df_s2id_naturais[colunas_PE].sum(axis=1)
)

In [80]:
df_impacto = (
    df_s2id_naturais
    .groupby('codigo_municipio')
    .agg(
        municipio=('Município', 'first'),
        DH_total=('DH_total', 'sum'),
        DM_total=('DM_total', 'sum'),
        PE_total=('PE_total', 'sum'),
        total_ocorrencias=('Protocolo', 'nunique')
    )
    .reset_index()
)

In [82]:

#iNTEGRAÇÃO DOS DADOS

df_impacto_completo = (
    df_demografia_limpo[
        ['codigo_municipio', 'municipio', 'populacao_total']
    ]
    .drop_duplicates('codigo_municipio')
    .merge(
        df_impacto.drop(columns='municipio'),
        on='codigo_municipio',
        how='left'
    )
)

colunas_impacto = [
    'DH_total', 'DM_total', 'PE_total', 'total_ocorrencias'
]

df_impacto_completo[colunas_impacto] = (
    df_impacto_completo[colunas_impacto].fillna(0)
)

df_impacto_completo['DH_per_1000'] = (
    df_impacto_completo['DH_total']
    / df_impacto_completo['populacao_total'] * 1000
)

df_impacto_completo['DM_per_capita'] = (
    df_impacto_completo['DM_total']
    / df_impacto_completo['populacao_total']
)

df_impacto_completo['PE_per_capita'] = (
    df_impacto_completo['PE_total']
    / df_impacto_completo['populacao_total']
)

In [90]:
#integração da componente I

def normalizar_log(coluna):
    valores = np.log1p(coluna.clip(lower=0))

    minimo = valores.min()
    maximo = valores.max()

    if maximo == minimo:
        return pd.Series(0.0, index=coluna.index)

    return (valores - minimo) / (maximo - minimo)


df_impacto_completo['DH_norm'] = normalizar_log(
    df_impacto_completo['DH_per_1000']
)

df_impacto_completo['DM_norm'] = normalizar_log(
    df_impacto_completo['DM_per_capita']
)

df_impacto_completo['PE_norm'] = normalizar_log(
    df_impacto_completo['PE_per_capita']
)

df_impacto_completo['I'] = (
    df_impacto_completo['DH_norm']
    + df_impacto_completo['DM_norm']
    + df_impacto_completo['PE_norm']
) / 3

display(
    df_impacto_completo.sort_values(
        'I', ascending=False
    )[['municipio', 'DH_norm', 'DM_norm', 'PE_norm', 'I']].head(15)
)

,municipio,DH_norm,DM_norm,PE_norm,I
306,Rio Bonito do Iguaçu,0.913620,0.837114,0.718055,0.822930
36,Boa Esperança do Iguaçu,0.942122,0.742218,0.732220,0.805520
97,Diamante D'Oeste,0.909045,0.722065,0.710184,0.780431
199,Manfrinópolis,0.982703,0.671141,0.686180,0.780008
210,Maripá,0.871132,0.698517,0.765635,0.778428
270,Pitanga,0.957381,0.733223,0.614543,0.768382
212,Marquinho,0.969172,0.669025,0.656374,0.764857
291,Querência do Norte,0.889719,0.638534,0.715010,0.747754
285,Prudentópolis,0.895672,0.647979,0.664974,0.736208
105,Espigão Alto do Iguaçu,0.926513,0.571260,0.703609,0.733794


In [91]:
# integração dos componentes V + F + I

# 399 municípios
df_ippdn = df_vulnerabilidade[
    ['codigo_municipio', 'municipio', 'V']
].copy()

# FREQUENCIA
df_ippdn = df_ippdn.merge(
    df_frequencia_completa[
        ['codigo_municipio', 'F', 'total_ocorrencias']
    ],
    on='codigo_municipio',
    how='left',
    validate='one_to_one'
)

# IMPACTO
df_ippdn = df_ippdn.merge(
    df_impacto_completo[
        ['codigo_municipio', 'I']
    ],
    on='codigo_municipio',
    how='left',
    validate='one_to_one'
)

# Verificar valores ausentes
print('Valores ausentes:')
print(df_ippdn[['V', 'F', 'I']].isna().sum())

Valores ausentes:
V    0
F    0
I    0
dtype: int64


In [93]:

# ------------------ CÁLCULO DO IPPDN ----------------------


# Pesos iguais
peso_V = 1/3
peso_F = 1/3
peso_I = 1/3

df_ippdn['IPPDN'] = (
    peso_V * df_ippdn['V'] +
    peso_F * df_ippdn['F'] +
    peso_I * df_ippdn['I']
)

# Ordenar municípios por prioridade
df_ippdn = df_ippdn.sort_values(
    'IPPDN',
    ascending=False
).reset_index(drop=True)

# Ranking
df_ippdn['posicao'] = (
    df_ippdn.index + 1
)


print('Total de municípios:', len(df_ippdn))
print('Índice mínimo:', df_ippdn['IPPDN'].min())
print('Índice máximo:', df_ippdn['IPPDN'].max())
print('Média:', df_ippdn['IPPDN'].mean())


Total de municípios: 399
Índice mínimo: 0.052589023903291246
Índice máximo: 0.7520907921992657
Média: 0.4002858693863438


In [94]:

# ------------------- CLASSIFICAÇÃO DOS GRAUS DE PRIORIDADE -----------------

limites = [
    -0.000001,
    0.20,
    0.40,
    0.60,
    0.80,
    1.000001
]

classes = [
    'Muito baixo',
    'Baixo',
    'Moderado',
    'Alto',
    'Muito alto'
]

df_ippdn['grau_prioridade'] = pd.cut(
    df_ippdn['IPPDN'],
    bins=limites,
    labels=classes,
    include_lowest=True
)

# Quantidade de municípios por classe
print(
    df_ippdn['grau_prioridade']
    .value_counts()
    .sort_index()
)

grau_prioridade
Muito baixo     75
Baixo           91
Moderado       203
Alto            30
Muito alto       0
Name: count, dtype: int64


In [95]:
df_ippdn.to_excel(
    'IPPDN_PR_resultados.xlsx',
    index=False
)

In [96]:
df_ippdn = df_ippdn.sort_values(
    'IPPDN',
    ascending=False
).reset_index(drop=True)

df_ippdn['posicao'] = range(1, len(df_ippdn) + 1)

print(df_ippdn.head(10))
print("Total de municípios:", len(df_ippdn))

  codigo_municipio               municipio         V         F  \
0          4115457               Marquinho  0.635214  0.856201   
1          4121000      Querência do Norte  0.636567  0.733452   
2          4107546  Espigão Alto do Iguaçu  0.721642  0.649561   
3          4113254                Laranjal  0.730283  0.598104   
4          4122503                Roncador  0.609908  0.768622   
5          4114351           Manfrinópolis  0.587732  0.649561   
6          4119608                 Pitanga  0.467503  0.768622   
7          4123857    Santa Maria do Oeste  0.674370  0.649561   
8          4110805                 Iretama  0.663875  0.598104   
9          4125100     São João do Triunfo  0.502837  0.733452   

   total_ocorrencias         I     IPPDN  posicao grau_prioridade  
0               12.0  0.764857  0.752091        1            Alto  
1                8.0  0.747754  0.705924        2            Alto  
2                6.0  0.733794  0.701666        3            Alto  
3

In [98]:
print(
    df_ippdn['grau_prioridade']
    .value_counts()
    .reindex([
        'Muito baixo',
        'Baixo',
        'Moderado',
        'Alto',
        'Muito alto'
    ], fill_value=0)
)

print(df_ippdn['IPPDN'].describe())

grau_prioridade
Muito baixo     75
Baixo           91
Moderado       203
Alto            30
Muito alto       0
Name: count, dtype: int64
count    399.000000
mean       0.400286
std        0.158232
min        0.052589
25%        0.256011
50%        0.423125
75%        0.519017
max        0.752091
Name: IPPDN, dtype: float64


In [99]:
df_ippdn['prioridade_quantiles'] = pd.qcut(
    df_ippdn['IPPDN'],
    q=5,
    labels=[
        'Muito baixo',
        'Baixo',
        'Moderado',
        'Alto',
        'Muito alto'
    ]
)

print(
    df_ippdn['prioridade_quantiles']
    .value_counts()
    .sort_index()
)

prioridade_quantiles
Muito baixo    80
Baixo          80
Moderado       79
Alto           80
Muito alto     80
Name: count, dtype: int64


In [101]:
#Elaboração da distribuição espacial

url = "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_municipais/municipio_2022/UFs/PR/PR_Municipios_2022.zip"

gdf = gpd.read_file(url)

print(gdf.columns.tolist())
print("Municípios na malha:", len(gdf))

['CD_MUN', 'NM_MUN', 'SIGLA_UF', 'AREA_KM2', 'geometry']
Municípios na malha: 399
